# Pipeline Runs Fact Table Loader

This notebook maintains the `warehouse.fact_pipeline_runs` fact table.

**Purpose**: Track ETL pipeline execution metrics and operational health

**Key Features**:
* One grain: One row per pipeline run
* Execution metrics (duration, records processed, status)
* Data quality metrics from each pipeline stage
* Run-level diagnostics for monitoring and alerting

**Sources**: 
* `audit.pipeline_run_log` (pipeline execution logs)
* `audit.pipeline_stage_metrics` (stage-level metrics)
* `workspace.warehouse.dim_date` (date dimension)

**Target**: `workspace.warehouse.fact_pipeline_runs`

In [0]:
%sql
-- Extract pipeline run execution data from actual audit table
CREATE OR REPLACE TEMP VIEW pipeline_runs_extract AS
SELECT 
  run_id as batch_id,
  pipeline_name,
  status as run_status,
  start_time as started_at,
  end_time as completed_at,
  CAST(runtime_seconds / 60.0 AS INT) as duration_minutes,
  rows_read as records_read,
  rows_written as records_written,
  0 as records_failed,  -- Not tracked in audit_pipeline_runs
  0 as records_quarantined,  -- Not tracked in audit_pipeline_runs
  error_message,
  environment as execution_user  -- Using environment as proxy
FROM workspace.audit.audit_pipeline_runs
WHERE run_id IS NOT NULL;

In [0]:
%sql
-- Aggregate DQ results per run (replaces stage metrics)
CREATE OR REPLACE TEMP VIEW stage_metrics_agg AS
SELECT 
  run_id as batch_id,
  COUNT(DISTINCT rule_name) as total_stages,  -- DQ rules as proxy for stages
  SUM(CASE WHEN failed_records = 0 THEN 1 ELSE 0 END) as successful_stages,
  SUM(CASE WHEN failed_records > 0 THEN 1 ELSE 0 END) as failed_stages,
  0 as total_records_in,  -- Not available in DQ results
  0 as total_records_out,  -- Not available in DQ results
  SUM(failed_records) as total_dq_failures
FROM workspace.audit.audit_dq_results
GROUP BY run_id;

In [0]:
%sql
-- Join runs with aggregated stage metrics
CREATE OR REPLACE TEMP VIEW runs_with_metrics AS
SELECT 
  r.*,
  COALESCE(s.total_stages, 0) as total_stages,
  COALESCE(s.successful_stages, 0) as successful_stages,
  COALESCE(s.failed_stages, 0) as failed_stages,
  COALESCE(s.total_records_in, 0) as total_records_in,
  COALESCE(s.total_records_out, 0) as total_records_out,
  COALESCE(s.total_dq_failures, 0) as total_dq_failures,
  -- Calculate data quality metrics
  CASE 
    WHEN r.records_read > 0 THEN 
      (r.records_written * 1.0 / r.records_read) * 100
    ELSE NULL
  END as success_rate_pct,
  CASE 
    WHEN r.records_read > 0 THEN 
      (r.records_failed * 1.0 / r.records_read) * 100
    ELSE NULL
  END as failure_rate_pct
FROM pipeline_runs_extract r
LEFT JOIN stage_metrics_agg s
  ON r.batch_id = s.batch_id;

In [0]:
%sql
-- Resolve to date dimension
CREATE OR REPLACE TEMP VIEW runs_with_date AS
SELECT 
  r.*,
  COALESCE(d.date_sk, -1) as run_date_sk
FROM runs_with_metrics r
LEFT JOIN workspace.warehouse.dim_date d
  ON DATE(r.started_at) = d.date_value;

In [0]:
%sql
-- Merge into fact table
MERGE INTO workspace.warehouse.fact_pipeline_runs target
USING (
  SELECT
    COALESCE(f.fact_pipeline_run_sk,
      ROW_NUMBER() OVER (ORDER BY r.batch_id) + COALESCE(max_sk, 0)) as fact_pipeline_run_sk,
    r.batch_id,
    r.pipeline_name,
    r.run_date_sk,
    r.run_status,
    r.started_at,
    r.completed_at,
    r.duration_minutes,
    r.records_read,
    r.records_written,
    r.records_failed,
    r.records_quarantined,
    r.total_stages,
    r.successful_stages,
    r.failed_stages,
    r.total_dq_failures,
    r.success_rate_pct,
    r.failure_rate_pct,
    r.error_message,
    r.execution_user
  FROM runs_with_date r
  LEFT JOIN workspace.warehouse.fact_pipeline_runs f
    ON r.batch_id = f.batch_id
  CROSS JOIN (
    SELECT COALESCE(MAX(fact_pipeline_run_sk), 0) as max_sk 
    FROM workspace.warehouse.fact_pipeline_runs
  ) max_keys
) source
ON target.batch_id = source.batch_id
WHEN MATCHED THEN UPDATE SET
  target.run_status = source.run_status,
  target.completed_at = source.completed_at,
  target.duration_minutes = source.duration_minutes,
  target.records_written = source.records_written,
  target.successful_stages = source.successful_stages,
  target.failed_stages = source.failed_stages,
  target.error_message = source.error_message
WHEN NOT MATCHED THEN INSERT (
  fact_pipeline_run_sk,
  batch_id,
  pipeline_name,
  run_date_sk,
  run_status,
  started_at,
  completed_at,
  duration_minutes,
  records_read,
  records_written,
  records_failed,
  records_quarantined,
  total_stages,
  successful_stages,
  failed_stages,
  total_dq_failures,
  success_rate_pct,
  failure_rate_pct,
  error_message,
  execution_user
) VALUES (
  source.fact_pipeline_run_sk,
  source.batch_id,
  source.pipeline_name,
  source.run_date_sk,
  source.run_status,
  source.started_at,
  source.completed_at,
  source.duration_minutes,
  source.records_read,
  source.records_written,
  source.records_failed,
  source.records_quarantined,
  source.total_stages,
  source.successful_stages,
  source.failed_stages,
  source.total_dq_failures,
  source.success_rate_pct,
  source.failure_rate_pct,
  source.error_message,
  source.execution_user
);

In [0]:
%sql
-- Validate pipeline runs fact table
SELECT 
  COUNT(*) as total_runs,
  COUNT(DISTINCT pipeline_name) as unique_pipelines,
  SUM(CASE WHEN run_status = 'SUCCESS' THEN 1 ELSE 0 END) as successful_runs,
  SUM(CASE WHEN run_status = 'FAILED' THEN 1 ELSE 0 END) as failed_runs,
  AVG(duration_minutes) as avg_duration_minutes,
  SUM(records_read) as total_records_read,
  SUM(records_written) as total_records_written,
  AVG(success_rate_pct) as avg_success_rate
FROM workspace.warehouse.fact_pipeline_runs;

-- Pipeline health by pipeline
SELECT 
  pipeline_name,
  COUNT(*) as run_count,
  SUM(CASE WHEN run_status = 'SUCCESS' THEN 1 ELSE 0 END) as success_count,
  AVG(duration_minutes) as avg_duration,
  AVG(success_rate_pct) as avg_success_rate,
  MAX(started_at) as last_run
FROM workspace.warehouse.fact_pipeline_runs
GROUP BY pipeline_name
ORDER BY last_run DESC;